In [ ]:
import sys
# Use an absolute path or relative path to the directory
sys.path.append("scripts")

import numpy as np
import pdb_voxelizier
import cnn_mlp_encoder
import jw_quantum_mapper
from scipy.optimize import minimize
from scipy.spatial.distance import squareform
from scipy.linalg import eigh
import sympy
import openfermion as op
import pyvista as pv
import torch
import cirq

In [ ]:
num_sites = 4
tensor = pdb_voxelizier.pdb_to_tensor('proteins/1ENH.pdb')
coefficients = cnn_mlp_encoder.get_hamiltonian(tensor, num_qubits=num_sites)
qubit_instructions = jw_quantum_mapper.apply_jw(coefficients,num_sites=num_sites)

# qubit_instructions

In [ ]:
def visualize_tensor(tensor):
    # [batch_size, channels, Depth, Height, Width]
    protein = tensor[0]
    
    protein = protein.to_dense().max(axis=0).values

    array = protein.numpy()
    grid = pv.wrap(array) # Automatically recognizes as a dataset
    grid.plot(jupyter_backend="client",volume=True)

torch_tensor = torch.from_numpy(tensor)
visualize_tensor(torch_tensor)

Test our hamiltonian through a VQE!

In [ ]:
# To convert our list of strings into a PauliSum, we need to loop through each instruction
def convert_qubit_operators_to_paulisum(qubit_operators:list):
    complete_pauli_string = op.QubitOperator()
    for i in qubit_operators:
  
        # Split up the operation into coefficient and the gates
        operation_list = i.split("*")
        
        # We grab the coefficient as a float and each operator as a single string   
        coef = float(operation_list[0])
        operators = operation_list[1].strip()

        # We construct qubit operators using OpenFermion
        complete_pauli_string += op.QubitOperator(operators,coef)
        
        # Convert string coefficient into float
        coef = float(operation_list[0])
    
    # Convert those qubit operators directly into a paulisum
    complete_pauli_string = op.qubit_operator_to_pauli_sum(complete_pauli_string)
    
    return complete_pauli_string

pauli_string = convert_qubit_operators_to_paulisum(qubit_instructions)

Get the ground state energy classically

In [ ]:
# Now, we need to get the ground state energy using OpenFermion

# First, we need to convert our coefficients into a matrix that can be processed by OpenFermion
# We can define a hamiltonian (hermitian) matrix using the condensed 1D coefficient matrix:
#  - The first 'num_sites' values map to individual site energies, which can be the diagonal values of the hermitian matrix
site_energies = coefficients[:num_sites]
print(f"Site energies: {site_energies}") 
# - The rest of the values are hopping integrals, and can be used to define the upper triangular of the hermitian matrix
#   (Which also defines the lower triangular, as hermitian matrices are symmetric)
hopping_integrals = coefficients[num_sites:]
print(f"Hopping integrals: {hopping_integrals}")

# To define the matrix, we use scipy.spatial.distance's squareform to get a Hermitian matrix
hamiltonian_matrix = squareform(hopping_integrals)

# From here, we fill the diagonals with site energies

np.fill_diagonal(hamiltonian_matrix,site_energies)
print("\nFinal Hamiltonian Matrix:")
print(hamiltonian_matrix)

# Now, we use numpy.linalg.eigh to calculate the ground state value of this matrix
lowest = min(eigh(hamiltonian_matrix)[0])
# lowest = op.get_ground_state(hamiltonian_matrix)[0]

print(f"\nGround State Energy: {lowest}")


Estimate ground state energy using VQE

In [ ]:
# Now, lets create our ansatz circuit. In this case I use a variation of the hardware efficient Ansatz (HEA) from the Lab 9 Knapsack problem example
# -----------------------------
# Parameterized circuit (ansatz)
# -----------------------------
qubits = list(pauli_string.qubits)
n = len(qubits)
symbols = sympy.symbols(f'theta(0:{n*3})')
print(type(symbols))
circuit = cirq.Circuit()

for l in range(3):  # 3 layers
    for i in range(n):
        circuit.append(cirq.rx(symbols[l * n + i])(qubits[i]))
    for i in range(n - 1):
        circuit.append(cirq.CZ(qubits[i], qubits[i + 1]))
print(circuit)

# Also taken from Lab 9 Knapsack problem example

# -----------------------------
# Expectation function
# -----------------------------

def expectation(params):
    resolver = cirq.ParamResolver({str(symbols[i]): params[i] for i in range(len(symbols))})
    result = simulator.simulate(circuit, resolver);

    # "i" for indice, "q" for qubit
    var = pauli_string.expectation_from_state_vector(
        result.final_state_vector,
        qubit_map={q: i for i, q in enumerate(qubits)}
    ).real
    # print(var)
    return var


Now we can run our circuit and attempt to converge on the ground state energy of the PauliSum.
There are a few optimizers we can choose from:
Type of solver. Should be one of:
    'Nelder-Mead'
    'Powell' 
    'CG' 
    'BFGS' 
    'Newton-CG' 
    'L-BFGS-B' 
    'TNC' 
    'COBYLA' 
    'COBYQA' 
    'SLSQP' 
    'trust-constr'
    'dogleg' 
    'trust-ncg' 
    'trust-exact' 
    'trust-krylov' 

In [ ]:
# -----------------------------
# VQE Optimization
# -----------------------------
simulator = cirq.Simulator()

# Random Angles calculated here!
x0 = np.random.uniform(0, 2 * np.pi, len(symbols))

res = minimize(expectation, x0=x0, method='COBYLA', options={'maxiter': 1000})

print("Optimal parameters:", res.x)
print("Minimum energy estimate:", res.fun)

# Now that we have our optimal parameters and minimum energy estimate, we run the circuit with those parameters to verify the output
resolver = cirq.ParamResolver({str(symbols[i]): res.x[i] for i in range(len(symbols))})
sample = simulator.simulate(circuit, resolver)

result = pauli_string.expectation_from_state_vector(
    sample.final_state_vector,
    qubit_map={q: i for i, q in enumerate(qubits)}
).real

print(f"The ground state energy from quantum circuit: {result}")
print(f"The ground state energy from diagnalization: {lowest}")